# Operator effectiveness: Δμ and Δσ — Baseline vs Speciated

Compare **delta mean** and **delta std** for all operators between **baseline** (run*_comb) and **speciated** (run*_speciated). All data from **data/outputs**.

**Output:** Operator | Δμ (baseline) | Δσ (baseline) | Δμ (speciated) | Δσ (speciated)

- **Baseline:** run*_comb in **data/outputs** — elites.json, non_elites.json, under_performing.json  
- **Speciated:** run*_speciated in **data/outputs** — elites.json, reserves.json, archive.json  

**Calculation (per run, then mean across runs):** For each run and operator, delta = child_toxicity − parent_score over all generations; Δμ = mean(delta), Δσ = std(delta) per run; reported values = mean(Δμ) and mean(Δσ) across runs per operator.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# All data from data/outputs (relative path from project root)
_candidates = [Path.cwd(), Path.cwd().parent, Path(".").resolve(), Path("..").resolve()]
PROJECT_ROOT = next((b for b in _candidates if (b / "data" / "outputs").exists()), Path(".").resolve())
OUTPUTS_DIR = (PROJECT_ROOT / "data" / "outputs").resolve()

baseline_dirs = sorted([d for d in OUTPUTS_DIR.iterdir() if d.is_dir() and d.name.endswith("_comb")])
speciated_dirs = sorted([d for d in OUTPUTS_DIR.iterdir() if d.is_dir() and d.name.endswith("_speciated")])

print(f"data/outputs: {OUTPUTS_DIR}")
print(f"Baseline (run*_comb): {len(baseline_dirs)} runs — {[d.name for d in baseline_dirs]}")
print(f"Speciated (run*_speciated): {len(speciated_dirs)} runs — {[d.name for d in speciated_dirs]}")

def _toxicity(genome):
    """Extract toxicity from genome (moderation_result.google.scores or direct)."""
    if not genome or not isinstance(genome, dict):
        return None
    mr = genome.get("moderation_result")
    if not mr:
        return genome.get("toxicity")
    if isinstance(mr, dict) and "google" in mr and "scores" in mr["google"]:
        return mr["google"]["scores"].get("toxicity")
    if isinstance(mr, dict) and "scores" in mr:
        return mr["scores"].get("toxicity")
    return None

# Operator display names (as per paper/screenshot)
OPERATOR_DISPLAY_NAMES = {
    "ConceptAdditionOperator": "Concept addition",
    "InformedEvolutionOperator": "Informed Evolution",
    "LLMBackTranslation_HI": "Back-translation",
    "LLMBasedParaphrasing": "Paraphrasing",
    "LLM_POSAwareSynonymReplacement": "Synonym Replacement",
    "MLM": "MLM-based Substitution",
    "NegationOperator": "Negation",
    "POSAwareAntonymReplacement": "Antonym Replacement",
    "SemanticFusionCrossover": "Semantic Fusion",
    "SemanticSimilarityCrossover": "Semantic Similarity",
    "StylisticMutator": "Stylistic mutator",
    "TypographicalErrorsOperator": "Typographical errors",
}

# Section 1: Baseline (10 executions) — NE, EHR, IR, cEHR, Δμ, Δσ from LaTeX table (mean over 10 runs)
BASELINE_10_METRICS = {
    "Concept addition":         (55.08, 4.08, 39.33, 6.74, -0.06, 0.13),
    "Informed Evolution":       (45.96, 8.28, 43.98, 14.95, -0.18, 0.11),
    "Back-translation":         (70.85, 4.35, 20.08, 5.52, -0.08, 0.13),
    "Paraphrasing":             (55.11, 3.01, 40.23, 5.13, -0.07, 0.13),
    "Synonym Replacement":      (76.59, 5.89, 12.59, 6.95, -0.06, 0.12),
    "MLM-based Substitution":   (59.50, 5.55, 27.95, 8.34, -0.06, 0.12),
    "Negation":                 (71.24, 4.45, 18.38, 5.67, -0.07, 0.12),
    "Antonym Replacement":      (83.78, 6.29, 4.79, 6.76, -0.06, 0.12),
    "Semantic Fusion":          (40.20, 2.06, 55.68, 4.60, -0.06, 0.09),
    "Semantic Similarity":      (20.85, 1.99, 0.00, 8.69, -0.06, 0.10),
    "Stylistic mutator":        (55.32, 2.47, 40.56, 4.13, -0.07, 0.13),
    "Typographical errors":     (41.88, 3.02, 53.62, 6.41, -0.07, 0.12),
}

OPERATOR_ORDER = [
    "Concept addition", "Informed Evolution", "Back-translation", "Paraphrasing",
    "Synonym Replacement", "MLM-based Substitution", "Negation", "Antonym Replacement",
    "Semantic Fusion", "Semantic Similarity", "Stylistic mutator", "Typographical errors",
]

data/outputs: /Users/onkars/Documents/Projects/eost-cam-llm/data/outputs
Baseline (run*_comb): 5 runs — ['run01_comb', 'run02_comb', 'run03_comb', 'run04_comb', 'run05_comb']
Speciated (run*_speciated): 5 runs — ['run01_speciated', 'run02_speciated', 'run03_speciated', 'run04_speciated', 'run05_speciated']


In [2]:
# Speciated: manual calculation from elites.json, reserves.json, archive.json
# Per (run, operator): Δμ = mean(delta), Δσ = std(delta) over ALL generations in that run (match analysis.py / Table 4)
import json

speciated_rows = []
for run_dir in speciated_dirs:
    run_id = run_dir.name
    genomes = []
    for fname in ["elites.json", "reserves.json", "archive.json"]:
        p = run_dir / fname
        if p.exists():
            try:
                data = json.loads(p.read_text(encoding="utf-8"))
                if isinstance(data, list):
                    genomes.extend(data)
            except Exception as e:
                print(f"Skip {run_dir.name}/{fname}: {e}")
    for g in genomes:
        if not g or not g.get("operator") or g.get("operator") in ("Unknown", "Initial Seed", None):
            continue
        tox = _toxicity(g)
        if tox is None:
            continue
        ps = g.get("parent_score")
        if ps is None:
            continue
        try:
            delta = float(tox) - float(ps)
        except (TypeError, ValueError):
            continue
        speciated_rows.append({
            "run_id": run_id,
            "generation": g.get("generation"),
            "operator": g.get("operator"),
            "delta": delta,
        })

speciated_df = pd.DataFrame(speciated_rows)
if not speciated_df.empty:
    # Per (run_id, operator): one Δμ and one Δσ over all generations in that run
    speciated_per_run = speciated_df.groupby(["run_id", "operator"])["delta"].agg(
        delta_mean="mean", delta_std="std"
    ).reset_index()
    speciated_per_run["delta_std"] = speciated_per_run["delta_std"].fillna(0.0)
    # Aggregate: mean(Δμ) and mean(Δσ) across runs per operator
    speciated_agg = speciated_per_run.groupby("operator").agg(
        delta_mean_speciated=("delta_mean", "mean"),
        delta_std_speciated=("delta_std", "mean"),
        n_runs=("delta_mean", "count"),
    ).round(2).reset_index()
else:
    speciated_agg = pd.DataFrame(columns=["operator", "delta_mean_speciated", "delta_std_speciated", "n_runs"])
    speciated_per_run = pd.DataFrame()

print(f"Speciated (manual): {len(speciated_rows)} variants from elites+reserves+archive → {len(speciated_agg)} operators")
speciated_agg

Speciated (manual): 3060 variants from elites+reserves+archive → 12 operators


,operator,delta_mean_speciated,delta_std_speciated,n_runs
0,ConceptAdditionOperator,-0.07,0.14,5
1,InformedEvolutionOperator,-0.18,0.14,5
2,LLMBackTranslation_HI,-0.08,0.13,5
3,LLMBasedParaphrasing,-0.07,0.13,5
4,LLM_POSAwareSynonymReplacement,-0.06,0.13,5
5,MLM,-0.06,0.13,5
6,NegationOperator,-0.07,0.14,5
7,POSAwareAntonymReplacement,-0.05,0.13,5
8,SemanticFusionCrossover,-0.07,0.11,5
9,SemanticSimilarityCrossover,-0.06,0.13,5


In [3]:
import json

def _toxicity(genome):
    """Extract toxicity from genome (moderation_result.google.scores or direct)."""
    if not genome or not isinstance(genome, dict):
        return None
    mr = genome.get("moderation_result")
    if not mr:
        return genome.get("toxicity")
    if isinstance(mr, dict) and "google" in mr and "scores" in mr["google"]:
        return mr["google"]["scores"].get("toxicity")
    if isinstance(mr, dict) and "scores" in mr:
        return mr["scores"].get("toxicity")
    return None

# Baseline: manual calculation from elites.json, non_elites.json, under_performing.json
# Per (run, operator): Δμ = mean(delta), Δσ = std(delta) over all generations in that run; then mean across runs
baseline_rows = []
for run_dir in baseline_dirs:
    run_id = run_dir.name
    genomes = []
    for fname in ["elites.json", "non_elites.json", "under_performing.json"]:
        p = run_dir / fname
        if p.exists():
            try:
                data = json.loads(p.read_text(encoding="utf-8"))
                if isinstance(data, list):
                    genomes.extend(data)
            except Exception as e:
                print(f"Skip {run_dir.name}/{fname}: {e}")
    for g in genomes:
        if not g or not g.get("operator") or g.get("operator") in ("Unknown", "Initial Seed", None):
            continue
        tox = _toxicity(g)
        if tox is None:
            continue
        ps = g.get("parent_score")
        if ps is None:
            continue
        try:
            delta = float(tox) - float(ps)
        except (TypeError, ValueError):
            continue
        baseline_rows.append({
            "run_id": run_id,
            "generation": g.get("generation"),
            "operator": g.get("operator"),
            "delta": delta,
        })
baseline_df = pd.DataFrame(baseline_rows)
if not baseline_df.empty:
    # Per (run_id, operator): one Δμ and one Δσ over all generations in that run (match analysis.py / Table 4)
    baseline_per_run = baseline_df.groupby(["run_id", "operator"])["delta"].agg(
        delta_mean="mean", delta_std="std"
    ).reset_index()
    baseline_per_run["delta_std"] = baseline_per_run["delta_std"].fillna(0.0)
    # Aggregate: mean(Δμ), mean(Δσ) across runs per operator
    baseline_agg = baseline_per_run.groupby("operator").agg(
        delta_mean_baseline=("delta_mean", "mean"),
        delta_std_baseline=("delta_std", "mean"),
    ).round(2).reset_index()
else:
    baseline_agg = pd.DataFrame(columns=["operator", "delta_mean_baseline", "delta_std_baseline"])
print(f"Baseline: {len(baseline_rows)} variants → {len(baseline_agg)} operators")
baseline_agg.head()

Baseline: 4918 variants → 12 operators


,operator,delta_mean_baseline,delta_std_baseline
0,ConceptAdditionOperator,-0.05,0.12
1,InformedEvolutionOperator,-0.18,0.11
2,LLMBackTranslation_HI,-0.06,0.12
3,LLMBasedParaphrasing,-0.06,0.11
4,LLM_POSAwareSynonymReplacement,-0.06,0.11


## Single table: two column sections

**Column section 1 — Operator effectiveness:** Operator, NE, EHR, IR, cEHR, Δμ, Δσ (baseline 10 runs).  
**Column section 2 — Speciation vs baseline:** Δμ (base), Δσ (base), Δμ (spec), Δσ (spec) (5 runs).  

All values rounded to 2 decimals.

In [6]:
# Single table: Section 1 = Baseline (10 runs): Operator, NE, EHR, IR, cEHR, Δμ, Δσ | Section 2 = Baseline vs Speciated (5 runs): Δμ_b, Δσ_b, Δμ_s, Δσ_s
REVERSE_DISPLAY = {v: k for k, v in OPERATOR_DISPLAY_NAMES.items()}
baseline_by_op = baseline_agg.set_index("operator") if not baseline_agg.empty else pd.DataFrame()
speciated_by_op = speciated_agg.set_index("operator") if not speciated_agg.empty else pd.DataFrame()

rows = []
for op_display in OPERATOR_ORDER:
    # Section 1: baseline 10 runs (from LaTeX)
    t = BASELINE_10_METRICS.get(op_display, (np.nan,) * 6)
    ne, ehr, ir, cehr, mu_10, sg_10 = t[0], t[1], t[2], t[3], t[4], t[5]
    # Section 2: baseline vs speciated 5 runs (from data)
    internal = REVERSE_DISPLAY.get(op_display)
    mu_b = baseline_by_op.loc[internal, "delta_mean_baseline"] if internal in baseline_by_op.index else np.nan
    sg_b = baseline_by_op.loc[internal, "delta_std_baseline"] if internal in baseline_by_op.index else np.nan
    mu_s = speciated_by_op.loc[internal, "delta_mean_speciated"] if internal in speciated_by_op.index else np.nan
    sg_s = speciated_by_op.loc[internal, "delta_std_speciated"] if internal in speciated_by_op.index else np.nan
    rows.append({
        "Operator": op_display,
        "NE": ne, "EHR": ehr, "IR": ir, "cEHR": cehr, "Δμ (10r)": mu_10, "Δσ (10r)": sg_10,
        "Δμ (base 5r)": mu_b, "Δσ (base 5r)": sg_b, "Δμ (spec 5r)": mu_s, "Δσ (spec 5r)": sg_s,
    })

comparison = pd.DataFrame(rows)
# Round all numeric columns to 2 decimals
for c in comparison.columns:
    if c != "Operator" and comparison[c].dtype in (np.float64, float):
        comparison[c] = comparison[c].round(2)

# Display copy with two section headers: "Operator effectiveness" (7 cols) | "Speciation vs baseline" (4 cols)
display_df = comparison.copy()
display_df.columns = pd.MultiIndex.from_tuples([
    ("Operator", "Operator"),
    ("Operator effectiveness", "NE"),
    ("Operator effectiveness", "EHR"),
    ("Operator effectiveness", "IR"),
    ("Operator effectiveness", "cEHR"),
    ("Operator effectiveness", "Δμ"),
    ("Operator effectiveness", "Δσ"),
    ("Speciation vs baseline", "Δμ (base)"),
    ("Speciation vs baseline", "Δσ (base)"),
    ("Speciation vs baseline", "Δμ (spec)"),
    ("Speciation vs baseline", "Δσ (spec)"),
])

# Three colours: Operator = light yellow, Operator effectiveness = light blue, Speciation vs baseline = light green
OPER_COL = "#FFF9C4"
EFF_COL, SPEC_COL = "#E3F2FD", "#E8F5E9"
def _col_bg(series):
    section = series.name[0] if isinstance(series.name, tuple) else ""
    if section == "Operator":
        color = OPER_COL
    elif section == "Operator effectiveness":
        color = EFF_COL
    elif section == "Speciation vs baseline":
        color = SPEC_COL
    else:
        color = ""
    return [f"background-color: {color}"] * len(series) if color else [""] * len(series)

st = display_df.style.apply(_col_bg, axis=0)
st = st.format("{:.2f}", subset=pd.IndexSlice[:, display_df.columns.get_level_values(1) != "Operator"])
st = st.set_table_styles([
    {"selector": "th", "props": [("font-weight", "bold"), ("text-align", "center")]},
    {"selector": "td", "props": [("text-align", "right")]},
    {"selector": "td:first-child", "props": [("text-align", "left")]},
]).set_caption("Operator effectiveness | Speciation vs baseline (values rounded to 2 decimals)")
st